# Nutrition CallBot — Colab Runtime
**Stack:** Gemini 2.5 Flash · E5-large-instruct · Qdrant local

In [ ]:
# CELL 1: Clone repo
!git clone https://github.com/enigmaticcat/legal-voice-callbot.git
%cd /content/legal-voice-callbot
!git pull


In [ ]:
# CELL 2: Cài đặt dependencies
# Brain + RAG
!pip install -q \
    google-genai>=1.0.0 \
    sentence-transformers>=3.0.0 \
    qdrant-client>=1.7.0 \
    fastapi>=0.115.0 \
    "uvicorn[standard]>=0.30.0" \
    python-dotenv>=1.0.0 \
    aiohttp pandas numpy

# TTS — VieNeu đã bundle trong repo (tts/vieneu/)
!pip install -q \
    neucodec>=0.0.4 \
    phonemizer>=3.3.0 \
    librosa>=0.11.0 \
    huggingface_hub>=0.20.0 \
    soundfile torch torchaudio

# llama-cpp-python (backbone GGUF cho TTS)
!pip install -q llama-cpp-python>=0.3.16


In [ ]:
# CELL 3: Config — đọc từ Colab Secrets (icon 🔑 bên trái)
# Secrets cần thêm: GEMINI_API_KEY
# Nếu dùng vLLM thay Gemini: để trống GEMINI_API_KEY, đặt LLM_BACKEND=openai
import os
from google.colab import userdata

# ── Chọn LLM backend: 'gemini' hoặc 'openai' (vLLM/Ollama) ──
LLM_BACKEND = "gemini"  # đổi thành 'openai' nếu dùng vLLM

GEMINI_API_KEY = userdata.get("GEMINI_API_KEY") if LLM_BACKEND == "gemini" else ""
os.environ["GEMINI_API_KEY"] = GEMINI_API_KEY

# Qdrant chạy local (Cell 4 khởi động)
QDRANT_URL     = "http://localhost:6333"
QDRANT_API_KEY = ""
os.environ["QDRANT_URL"]     = QDRANT_URL
os.environ["QDRANT_API_KEY"] = QDRANT_API_KEY

# Tạo .env cho brain server
env_content = f"""GEMINI_API_KEY={GEMINI_API_KEY}
QDRANT_URL={QDRANT_URL}
QDRANT_API_KEY={QDRANT_API_KEY}
BRAIN_PORT=50052
TTS_URL=http://localhost:50053
LLM_BACKEND={LLM_BACKEND}
LLM_BASE_URL=http://localhost:8000/v1
LLM_MODEL=Qwen/Qwen3-4B-Instruct-2507
"""
with open("/content/legal-voice-callbot/nutrition-callbot/.env", "w") as f:
    f.write(env_content)

print(f"LLM_BACKEND   : {LLM_BACKEND}")
if LLM_BACKEND == 'gemini':
    print(f"GEMINI_API_KEY: {GEMINI_API_KEY[:8]}...")
print(f"QDRANT_URL    : {QDRANT_URL}")
print(".env          : OK")


In [ ]:
# CELL 4: Khởi động Qdrant local
import subprocess, time

!curl -sL https://github.com/qdrant/qdrant/releases/latest/download/qdrant-x86_64-unknown-linux-musl.tar.gz \n
    | tar -xz

qdrant_proc = subprocess.Popen(
    ["/content/qdrant"],
    stdout=open("qdrant.log", "w"),
    stderr=subprocess.STDOUT,
)
time.sleep(5)

import requests
r = requests.get("http://localhost:6333/healthz")
print(f"Qdrant: {r.status_code} (PID={qdrant_proc.pid})")


In [ ]:
# CELL 4b: [TÙY CHỌN] Khởi động vLLM thay cho Gemini API
# Chỉ chạy cell này nếu LLM_BACKEND = 'openai' ở Cell 3
# Yêu cầu: GPU A100/L4 (Colab Pro+), bỏ qua nếu dùng Gemini

import os
if os.environ.get("LLM_BACKEND", "gemini") != "openai":
    print("LLM_BACKEND=gemini → bỏ qua cell này.")
else:
    !pip install -q vllm

    import subprocess, time, requests, sys

    os.system("pkill -f vllm.entrypoints 2>/dev/null")
    time.sleep(2)

    %cd /content/legal-voice-callbot/nutrition-callbot/brain

    vllm_proc = subprocess.Popen([
        sys.executable, "-m", "vllm.entrypoints.openai.api_server",
        "--model",                    "Qwen/Qwen3-4B-Instruct-2507",
        "--speculative-model",        "Qwen/Qwen3-0.6B",
        "--port",                     "8000",
        "--max-model-len",            "4096",
        "--gpu-memory-utilization",   "0.8",
    ], stdout=open("/content/vllm.log", "w"), stderr=subprocess.STDOUT)

    print("Chờ vLLM sẵn sàng (tải model, có thể mất 3-5 phút)...")
    for i in range(120):
        try:
            if requests.get("http://localhost:8000/health", timeout=2).status_code == 200:
                print(f"vLLM ready! ({i*5}s, PID={vllm_proc.pid})")
                break
        except: pass
        time.sleep(5)
        if i == 119: print("TIMEOUT — xem /content/vllm.log")

    !tail -15 /content/vllm.log


In [ ]:
# CELL 5: Mount Google Drive + restore Qdrant snapshot
# Yêu cầu: file snapshot đã được tải từ Qdrant Cloud dashboard
#   Qdrant Cloud → Collection → Snapshots → Download → upload lên Google Drive
# Sau đó cập nhật SNAPSHOT_PATH bên dưới cho đúng tên file của bạn

from google.colab import drive
drive.mount("/content/drive")

# ⚠️ Cập nhật đường dẫn snapshot của bạn tại đây:
SNAPSHOT_PATH = "/content/drive/MyDrive/Nutrition data/nutrition_articles-2744933042503761-2026-03-30-08-11-07.snapshot"
COLLECTION    = "nutrition_articles"

import requests, os

if not os.path.exists(SNAPSHOT_PATH):
    raise FileNotFoundError(f"Không tìm thấy snapshot: {SNAPSHOT_PATH}
"
        "Hướng dẫn: Qdrant Cloud → Chọn collection → Snapshots → Download
"
        "Sau đó upload file .snapshot lên Google Drive và cập nhật SNAPSHOT_PATH")

# Xóa collection cũ nếu tồn tại
requests.delete(f"http://localhost:6333/collections/{COLLECTION}")

# Upload snapshot
print("Uploading snapshot (có thể mất 1-2 phút)...")
r = requests.post(
    f"http://localhost:6333/collections/{COLLECTION}/snapshots/upload?priority=snapshot",
    files={"snapshot": open(SNAPSHOT_PATH, "rb")},
    timeout=300,
)
print(f"Status: {r.status_code}")
assert r.status_code == 200, f"Upload thất bại: {r.text}"

# Kiểm tra
info = requests.get(f"http://localhost:6333/collections/{COLLECTION}").json()
print(f"Collection: {COLLECTION} | vectors: {info['result']['vectors_count']}")


In [ ]:
# CELL 6: Khởi động Brain server (Gemini)
import subprocess, time, requests, sys

PROJECT = "/content/legal-voice-callbot/nutrition-callbot"

# Kill server cũ nếu có
import os; os.system("pkill -f brain.server 2>/dev/null")
time.sleep(2)

brain_proc = subprocess.Popen(
    [sys.executable, "-m", "brain.server"],
    cwd=PROJECT,
    stdout=open("brain.log", "w"),
    stderr=subprocess.STDOUT,
)

# Chờ server sẵn sàng
for i in range(60):
    try:
        if requests.get("http://localhost:50052/health", timeout=2).status_code == 200:
            print(f"Brain server OK (PID={brain_proc.pid})")
            break
    except: pass
    time.sleep(2)
    if i == 59: print("TIMEOUT — xem brain.log")

# In 10 dòng log cuối
!tail -10 /content/brain.log


In [ ]:
# CELL 7: Test single query
import requests, json, time

QUERY = "Thưa bác sĩ, tôi muốn hỏi có nên uống Omega 3-6-9 mỗi ngày hay không?"

t0 = time.time()
r = requests.post(
    "http://localhost:50052/think/stream",
    json={"query": QUERY, "session_id": "test"},
    stream=True,
)

full_text = ""
ttft = None
for line in r.iter_lines():
    if not line: continue
    data = json.loads(line.decode())
    if data.get("text") and not data.get("is_final"):
        if ttft is None:
            ttft = time.time() - t0
        full_text += data["text"]
    if data.get("is_final"):
        timing = data.get("timing", {})

print(f"Query : {QUERY}")
print(f"Answer: {full_text}")
print(f"TTFT  : {ttft:.2f}s | Total: {time.time()-t0:.2f}s")


In [ ]:
# CELL 8: Batch evaluation trên thucuc_qa (255 câu)
import requests, json, time, os, pandas as pd, numpy as np, matplotlib.pyplot as plt, seaborn as sns
sns.set_theme(style="whitegrid")

# Giới hạn số mẫu để tránh quá tải (đổi về None nếu muốn chạy full)
EVAL_LIMIT = 80          # số câu hỏi tối đa để evaluate
PLOT_SAMPLE = 500        # lấy mẫu tối đa khi vẽ/percentile để giảm RAM
SLEEP_BETWEEN_REQS = 0.4 # nghỉ giữa các câu để giảm tải server
QA_PATH  = "/content/legal-voice-callbot/evaluation/thucuc_qa.jsonl"
PLOT_DIR = "/content/drive/MyDrive/Nutrition data/eval_plots"
OUT      = "/content/drive/MyDrive/Nutrition data/eval_results.jsonl"
os.makedirs(PLOT_DIR, exist_ok=True)

results = []
with open(QA_PATH, encoding="utf-8") as f:
    records = [json.loads(l) for l in f if l.strip()]
if EVAL_LIMIT:
    records = records[:EVAL_LIMIT]

print(f"Bắt đầu evaluate {len(records)} câu...")

for i, rec in enumerate(records):
    t0 = time.time()
    try:
        r = requests.post(
            "http://localhost:50052/think/stream",
            json={"query": rec["question"], "session_id": f"eval_{i}"},
            stream=True, timeout=60,
        )
        answer = ""
        contexts = []
        timing = {}
        ttft_s = None
        for line in r.iter_lines():
            if not line:
                continue
            d = json.loads(line.decode())
            if d.get("text") and not d.get("is_final"):
                if ttft_s is None:
                    ttft_s = time.time() - t0
                answer += d["text"]
            if d.get("is_final"):
                contexts = d.get("contexts", [])
                timing   = d.get("timing", {})
        results.append({
            "question"    : rec["question"],
            "answer"      : answer,
            "ground_truth": rec["answer"],
            "contexts"    : contexts,
            "timing"      : timing,
            "ttft_s"      : round(ttft_s or 0, 3),
            "latency_s"   : round(time.time() - t0, 2),
        })
    except Exception as e:
        results.append({"question": rec["question"], "error": str(e)})
    finally:
        time.sleep(SLEEP_BETWEEN_REQS)  # giảm tải giữa các câu

    if (i + 1) % 25 == 0:
        print(f"  {i+1}/{len(records)} done")

# Lưu kết quả
with open(OUT, "w", encoding="utf-8") as f:
    for r in results:
        f.write(json.dumps(r, ensure_ascii=False) + "\n")

errors  = sum(1 for r in results if "error" in r)
avg_lat = sum(r.get("latency_s", 0) for r in results if "error" not in r) / max(1, len(results) - errors)
print(f"Done: {len(results)} | Errors: {errors} | Avg latency: {avg_lat:.1f}s")
print(f"Saved JSONL: {OUT}")

# Vẽ biểu đồ và lưu
df = pd.DataFrame(results)
if "error" not in df.columns:
    df["error"] = pd.NA
df_ok = df[df["error"].isna()].copy()
df_ok["latency_ms"] = df_ok["latency_s"].astype(float) * 1000
df_ok["ttft_ms"]    = df_ok["ttft_s"].astype(float) * 1000
df_ok["rag_ms"]     = df_ok["timing"].apply(lambda t: (t or {}).get("rag_ms"))
df_ok["expand_ms"]  = df_ok["timing"].apply(lambda t: (t or {}).get("expand_ms"))
df_ok["llm_ttft_ms"] = df_ok["timing"].apply(lambda t: (t or {}).get("llm_ttft_ms"))

df_ok["tokens"] = df_ok["answer"].fillna("").apply(lambda x: len(str(x).split()))
df_ok["token_speed_tps"] = df_ok.apply(lambda r: (r["tokens"] / r["latency_s"]) if r.get("latency_s") else np.nan, axis=1)

def downsample(series, n=PLOT_SAMPLE):
    if len(series) > n:
        return series.sample(n, random_state=42)
    return series

def plot_hist(series, title, xlabel, fname):
    clean = downsample(series.dropna())
    if clean.empty:
        print(f"Bỏ qua {title} (không đủ dữ liệu)")
        return
    plt.figure(figsize=(7, 4))
    sns.histplot(clean, bins=30, kde=True)
    plt.title(title)
    plt.xlabel(xlabel)
    plt.ylabel("Count")
    plt.tight_layout()
    path = f"{PLOT_DIR}/{fname}"
    plt.savefig(path, dpi=150)
    plt.close()
    print(f"Saved plot: {path}")

plot_hist(df_ok["latency_ms"], "Latency (ms)", "Latency (ms)", "latency_hist.png")
plot_hist(df_ok["ttft_ms"], "TTFT (ms)", "TTFT (ms)", "ttft_hist.png")
plot_hist(df_ok["rag_ms"], "RAG Retrieval (ms)", "rag_ms", "rag_hist.png")
plot_hist(df_ok["expand_ms"], "Query Expansion (ms)", "expand_ms", "expand_hist.png")
plot_hist(df_ok["llm_ttft_ms"], "LLM TTFT (ms)", "llm_ttft_ms", "llm_ttft_hist.png")
plot_hist(df_ok["token_speed_tps"], "Token speed (tokens/s)", "tokens/s", "token_speed_hist.png")

# In percentile p50/90/95/99 cho TTFT, Retrieval, Token speed
def percentile_stats(series):
    clean = downsample(series.dropna().astype(float))
    if clean.empty:
        return None
    return {p: float(np.percentile(clean, p)) for p in [50, 90, 95, 99]}

def print_pct(label, series, unit=""):
    stats = percentile_stats(series)
    if not stats:
        print(f"{label}: không đủ dữ liệu")
        return None
    print(
        f"{label:<22} p50={stats[50]:.1f}{unit}  p90={stats[90]:.1f}{unit} "
        f"p95={stats[95]:.1f}{unit}  p99={stats[99]:.1f}{unit}  n={len(series.dropna())}"
    )
    return stats

summary = {
    "ttft_ms": print_pct("TTFT (ms)", df_ok["ttft_ms"], "ms"),
    "retrieval_ms": print_pct("Retrieval (ms)", df_ok["rag_ms"], "ms"),
    "token_speed_tps": print_pct("Token speed", df_ok["token_speed_tps"], " t/s"),
}

with open(f"{PLOT_DIR}/latency_summary.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)
print(f"Saved summary: {PLOT_DIR}/latency_summary.json")

print(f"Plots folder: {PLOT_DIR}")

In [ ]:
# CELL 9: Cài đặt Evaluation dependencies
# VieNeu-TTS đã được bundle sẵn trong repo tại nutrition-callbot/tts/vieneu/
# Không cần clone thêm

!pip install -q \
    soundfile \
    "ragas>=0.2.0" \
    langchain-google-genai \
    "faster-whisper>=1.0.0" \
    utmos


In [ ]:
# CELL 10: Khởi động TTS server (VieNeu bundled trong repo)
import subprocess, sys, time, requests, os

PROJECT = "/content/legal-voice-callbot/nutrition-callbot"
os.system("pkill -f tts.server 2>/dev/null")
time.sleep(1)

# VieNeu đã bundle tại tts/vieneu/ — không cần PYTHONPATH ngoài
tts_env = {
    **os.environ,
    "TTS_PORT": "50053",
}

tts_proc = subprocess.Popen(
    [sys.executable, "-m", "uvicorn", "tts.server:app",
     "--host", "0.0.0.0", "--port", "50053"],
    cwd=PROJECT,
    stdout=open("/content/tts.log", "w"),
    stderr=subprocess.STDOUT,
    env=tts_env,
)

print("Đang chờ TTS server (lần đầu tải model ~2-3 phút)...")
for i in range(90):
    try:
        if requests.get("http://localhost:50053/health", timeout=2).status_code == 200:
            print(f"✅ TTS server OK (PID={tts_proc.pid})")
            break
    except: pass
    time.sleep(3)
    if i % 10 == 9: print(f"  ...{(i+1)*3}s")
    if i == 89: print("❌ TIMEOUT — xem tts.log")

!tail -20 /content/tts.log


In [ ]:
# CELL 11: Test end-to-end pipeline (Brain → TTS)
import requests, json, time, soundfile as sf, io
import IPython.display as ipd

QUERY = "Thưa bác sĩ, tôi muốn hỏi có nên uống Omega 3-6-9 mỗi ngày hay không?"

# 1. Lấy câu trả lời từ Brain (Gemini + RAG)
t_brain = time.perf_counter()
r = requests.post(
    "http://localhost:50052/think",
    json={"query": QUERY, "session_id": "e2e_test"},
)
brain_data = r.json()
answer = brain_data["text"]
brain_s = time.perf_counter() - t_brain
print(f"Brain: {brain_s:.2f}s")
print(f"Answer ({len(answer)} ký tự):\n{answer[:300]}...\n")

# 2. Tổng hợp giọng nói qua VieneuTTS
t_tts = time.perf_counter()
tts_r = requests.post(
    "http://localhost:50053/speak",
    json={"text": answer},
    timeout=120,
)
tts_s = time.perf_counter() - t_tts

ttfb_ms = float(tts_r.headers.get("X-TTFB-ms", 0))
rtf     = float(tts_r.headers.get("X-RTF", 0))
dur_s   = float(tts_r.headers.get("X-Duration-s", 0))

print(f"TTS: {tts_s:.2f}s | TTFB: {ttfb_ms:.0f}ms | RTF: {rtf:.3f} | Dài: {dur_s:.1f}s")
print(f"Tổng pipeline: {brain_s + tts_s:.2f}s")

# Phát audio trong Colab
audio, sr = sf.read(io.BytesIO(tts_r.content))
print(f"\nAudio ({sr}Hz, {len(audio)/sr:.1f}s):")
ipd.display(ipd.Audio(audio, rate=sr))


In [ ]:
# CELL 12: RAGAs Evaluation (Context Precision, Recall, Faithfulness,
#           Answer Relevancy, Factual Correctness)
import json, os

# Load eval results (từ Cell 8)
results = []
with open("/content/drive/MyDrive/Nutrition data/eval_results.jsonl") as f:
    for line in f:
        if line.strip():
            results.append(json.loads(line))

ok = [r for r in results if "error" not in r and r.get("contexts")]
print(f"Records hợp lệ (có contexts): {len(ok)}/{len(results)}")

from ragas import evaluate, EvaluationDataset, SingleTurnSample
from ragas.metrics import (
    LLMContextPrecisionWithReference,
    ContextRecall,
    Faithfulness,
    ResponseRelevancy,
    FactualCorrectness,
)
from ragas.llms import LangchainLLMWrapper
from langchain_google_genai import ChatGoogleGenerativeAI

llm = LangchainLLMWrapper(ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    google_api_key=os.environ["GEMINI_API_KEY"],
))

EVAL_N = 50  # dùng 50 mẫu để tiết kiệm quota
samples = [
    SingleTurnSample(
        user_input=r["question"],
        response=r["answer"],
        reference=r["ground_truth"],
        retrieved_contexts=r["contexts"],
    )
    for r in ok[:EVAL_N]
]
dataset = EvaluationDataset(samples=samples)

result = evaluate(
    dataset=dataset,
    metrics=[
        LLMContextPrecisionWithReference(),
        ContextRecall(),
        Faithfulness(),
        ResponseRelevancy(),
        FactualCorrectness(),
    ],
    llm=llm,
)
print(result)

OUT = "/content/drive/MyDrive/Nutrition data/ragas_results.csv"
result.to_pandas().to_csv(OUT, index=False)
print(f"Saved: {OUT}")


In [ ]:
# CELL 13: Latency Analysis — p50 / p90 / p95 / p99
import json, numpy as np

results = []
with open("/content/drive/MyDrive/Nutrition data/eval_results.jsonl") as f:
    for line in f:
        if line.strip():
            results.append(json.loads(line))

ok = [r for r in results if "error" not in r]

def pct_row(label, vals_ms):
    a = np.array(vals_ms)
    row = f"{label:<22s}"
    for p in [50, 90, 95, 99]:
        row += f"  p{p}={np.percentile(a,p):.0f}ms"
    row += f"  mean={a.mean():.0f}ms  n={len(a)}"
    print(row)

print("=" * 70)
print(f"{'Metric':<22s}  {'p50':>8s}  {'p90':>8s}  {'p95':>8s}  {'p99':>8s}  mean")
print("=" * 70)

latencies_ms = [r["latency_s"] * 1000 for r in ok]
pct_row("Total latency", latencies_ms)

ttfts_ms = [r["ttft_s"] * 1000 for r in ok if r.get("ttft_s")]
if ttfts_ms:
    pct_row("TTFT", ttfts_ms)

rag_ms = [r["timing"]["rag_ms"] for r in ok if r.get("timing", {}).get("rag_ms")]
if rag_ms:
    pct_row("RAG retrieval", rag_ms)

expand_ms = [r["timing"]["expand_ms"] for r in ok if r.get("timing", {}).get("expand_ms")]
if expand_ms:
    pct_row("Query expansion", expand_ms)

llm_ms = [r["timing"]["llm_ttft_ms"] for r in ok if r.get("timing", {}).get("llm_ttft_ms")]
if llm_ms:
    pct_row("LLM TTFT", llm_ms)

print("=" * 70)
errors = len(results) - len(ok)
print(f"Total records: {len(results)} | Errors: {errors} | Success: {len(ok)}")


In [ ]:
# CELL 14: TTS Quality Evaluation
#   - UTMOS: naturalness MOS proxy (không cần human judge)
#   - WER via faster-whisper: intelligibility
#   - TTFB + RTF: latency percentiles
import requests, numpy as np, soundfile as sf, io, json, time, torch

# Load 20 câu trả lời mẫu
results = []
with open("/content/drive/MyDrive/Nutrition data/eval_results.jsonl") as f:
    for line in f:
        if line.strip():
            results.append(json.loads(line))
TEST_N = 20
test_samples = [r for r in results if "error" not in r and r.get("answer")][:TEST_N]

# ── 1. Tổng hợp audio + đo TTFB / RTF ──────────────────────────────────────
tts_stats = []
print(f"Synthesizing {len(test_samples)} samples...")
for i, rec in enumerate(test_samples):
    text = rec["answer"][:400]  # giới hạn độ dài
    r = requests.post("http://localhost:50053/speak", json={"text": text}, timeout=120)
    audio, sr = sf.read(io.BytesIO(r.content))
    tts_stats.append({
        "audio": audio, "sr": sr, "text": text,
        "ttfb_ms": float(r.headers.get("X-TTFB-ms", 0)),
        "rtf":     float(r.headers.get("X-RTF", 0)),
        "dur_s":   float(r.headers.get("X-Duration-s", 0)),
    })
    if (i+1) % 5 == 0: print(f"  {i+1}/{len(test_samples)}")

ttfbs = [s["ttfb_ms"] for s in tts_stats]
rtfs  = [s["rtf"]     for s in tts_stats]
print("\n=== TTS Latency ===")
for label, vals in [("TTFB (ms)", ttfbs), ("RTF", rtfs)]:
    a = np.array(vals)
    print(f"{label:<12s} p50={np.percentile(a,50):.1f}  p90={np.percentile(a,90):.1f}  p99={np.percentile(a,99):.1f}  mean={a.mean():.1f}")

# ── 2. UTMOS MOS (naturalness) ───────────────────────────────────────────────
try:
    import utmos
    predictor = utmos.score.load_model()
    mos_scores = []
    for s in tts_stats:
        wave_t = torch.FloatTensor(s["audio"]).unsqueeze(0)
        score  = predictor.score(wave_t, s["sr"])
        mos_scores.append(score.item())
    print(f"\nUTMOS MOS:  {np.mean(mos_scores):.3f} ± {np.std(mos_scores):.3f}")
    print(f"  min={min(mos_scores):.3f}  max={max(mos_scores):.3f}")
except Exception as e:
    print(f"\nUTMOS: {e}")

# ── 3. WER via faster-whisper (intelligibility) ──────────────────────────────
try:
    from faster_whisper import WhisperModel

    def word_error_rate(ref: str, hyp: str) -> float:
        import difflib
        r, h = ref.lower().split(), hyp.lower().split()
        ops   = difflib.SequenceMatcher(None, r, h).get_opcodes()
        edits = sum(max(o[2]-o[1], o[4]-o[3]) for o in ops if o[0] != "equal")
        return edits / max(1, len(r))

    whisper = WhisperModel("large-v3", device="cuda", compute_type="float16")
    wers = []
    for s in tts_stats[:10]:
        segs, _ = whisper.transcribe(s["audio"], language="vi")
        hyp = " ".join(seg.text for seg in segs)
        wers.append(word_error_rate(s["text"], hyp))
    print(f"WER (Intelligibility): {np.mean(wers):.3f} ± {np.std(wers):.3f}")
    print(f"  (WER thấp hơn = nghe rõ hơn)")
except Exception as e:
    print(f"WER: {e}")


In [ ]:
# CELL 15: RAGAS (Gemini API hoặc Vertex AI, one-cell)
import json, os
from ragas import evaluate, EvaluationDataset, SingleTurnSample

# Hỗ trợ cả ragas mới (collections) và legacy
try:
    from ragas.metrics.collections import (
        ContextPrecision,
        ContextRecall,
        Faithfulness,
        AnswerRelevancy,
        FactualCorrectness,
    )
    LLMContextPrecisionWithReference = ContextPrecision  # alias tương đương
    ResponseRelevancy = AnswerRelevancy
    METRIC_SRC = "collections"
except Exception:
    try:
        from ragas.metrics import (
            LLMContextPrecisionWithReference,
            ContextRecall,
            Faithfulness,
            ResponseRelevancy,
            FactualCorrectness,
        )
        METRIC_SRC = "legacy"
    except Exception as e:
        raise ImportError(
            "Không import được metrics từ ragas. Cài/upgrade: !pip install -q ragas"
        ) from e

from ragas.llms import LangchainLLMWrapper
try:
    from langchain_google_genai import ChatGoogleGenerativeAI
except ModuleNotFoundError:
    raise ModuleNotFoundError(
        "Thiếu langchain-google-genai. Chạy !pip install -q langchain-google-genai google-genai ragas"
    )

EVAL_RESULTS = "/content/drive/MyDrive/Nutrition data/eval_results.jsonl"
OUT_CSV     = "/content/drive/MyDrive/Nutrition data/ragas_results.csv"
EVAL_N      = 50   # số mẫu đánh giá, đổi None để dùng toàn bộ

# Chọn backend (mặc định dùng API key). Nếu USE_VERTEX=True nhưng thiếu env thì tự fallback.
USE_VERTEX   = False  # đặt True nếu muốn Vertex
MODEL        = "gemini-2.5-flash"

records = []
with open(EVAL_RESULTS) as f:
    for line in f:
        if line.strip():
            records.append(json.loads(line))

ok = [r for r in records if "error" not in r and r.get("contexts")]
print(f"Records hợp lệ: {len(ok)}/{len(records)}")
subset = ok if EVAL_N is None else ok[:EVAL_N]
print(f"Đánh giá {len(subset)} mẫu")

samples = [
    SingleTurnSample(
        user_input=r["question"],
        response=r["answer"],
        reference=r["ground_truth"],
        retrieved_contexts=r["contexts"],
    )
    for r in subset
]
dataset = EvaluationDataset(samples=samples)

if USE_VERTEX:
    PROJECT  = os.environ.get("VERTEX_PROJECT")
    LOCATION = os.environ.get("VERTEX_LOCATION", "us-central1")
    if not PROJECT:
        print("USE_VERTEX=True nhưng thiếu VERTEX_PROJECT → fallback sang GEMINI_API_KEY")
        USE_VERTEX = False
    else:
        llm_raw = ChatGoogleGenerativeAI(
            model=MODEL,
            project=PROJECT,
            location=LOCATION,
        )

if not USE_VERTEX:
    API_KEY = os.environ.get("GEMINI_API_KEY")
    if not API_KEY:
        raise ValueError("Thiếu GEMINI_API_KEY trong env (Colab Secrets)")
    llm_raw = ChatGoogleGenerativeAI(
        model=MODEL,
        google_api_key=API_KEY,
    )

llm = LangchainLLMWrapper(llm_raw)

# Khởi tạo metrics theo nguồn import
def build_metrics():
    if METRIC_SRC == "collections":
        return [
            LLMContextPrecisionWithReference(llm=llm),
            ContextRecall(llm=llm),
            Faithfulness(llm=llm),
            ResponseRelevancy(llm=llm),
            FactualCorrectness(llm=llm),
        ]
    return [
        LLMContextPrecisionWithReference(),
        ContextRecall(),
        Faithfulness(),
        ResponseRelevancy(),
        FactualCorrectness(),
    ]

metrics = build_metrics()
result = evaluate(
    dataset=dataset,
    metrics=metrics,
    llm=llm,
)
print(result)

result.to_pandas().to_csv(OUT_CSV, index=False)
print(f"Saved: {OUT_CSV}")